# Acne Segmentation & Severity Classification
**Dataset:** ACNE04-v2 | **Model:** Hybrid ResNet-50 + Dilated Decoder + AMFM

> Sebelum mulai: **Runtime → Change runtime type → T4 GPU → Save**

---
## 1. Cek GPU

In [ ]:
import torch
print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('⚠️  GPU belum aktif! Runtime → Change runtime type → T4 GPU')

---
## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive terhubung')

---
## 3. Clone Repo dari GitHub

> Ganti `GITHUB_USERNAME` dan `REPO_NAME` dengan milik kamu.

In [ ]:
# ── Ganti dengan URL repo GitHub kamu ────────────────────
GITHUB_URL = 'https://github.com/GITHUB_USERNAME/REPO_NAME.git'
REPO_DIR   = '/content/acne-seg'
# ─────────────────────────────────────────────────────────

import os
if os.path.exists(REPO_DIR):
    # Sudah ada → pull saja (ambil update terbaru)
    !git -C "{REPO_DIR}" pull
    print('✅ Repo di-pull (update terbaru)')
else:
    # Belum ada → clone
    !git clone "{GITHUB_URL}" "{REPO_DIR}"
    print('✅ Repo berhasil di-clone')

!ls "{REPO_DIR}/scripts/"

---
## 4. Install Dependencies

In [ ]:
%%capture
!pip install torch torchvision pillow opencv-python-headless \
             numpy matplotlib pandas tqdm kagglehub gdown
print('✅ Dependencies terinstall')

---
## 5. Download Dataset

Pilih **salah satu** Option di bawah.

In [ ]:
# ── OPTION A: Kaggle (Rekomendasi) ───────────────────────
import kagglehub
KAGGLE_PATH = kagglehub.dataset_download('karmagames/acne04-v2')
print('✅ Kaggle dataset di:', KAGGLE_PATH)
!ls "{KAGGLE_PATH}"

In [ ]:
# ── OPTION B: Google Drive folder ────────────────────────
# Uncomment jika pakai option ini
# import gdown
# gdown.download_folder(
#     'https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ',
#     output='/content/acne_images/',
#     quiet=False
# )
# KAGGLE_PATH = '/content/acne_images'
# print('✅ Dataset di:', KAGGLE_PATH)

---
## 6. Clone Anotasi v2 dari GitHub

In [ ]:
import os
ANNOT_REPO = '/content/acne04v2'

if os.path.exists(ANNOT_REPO):
    !git -C "{ANNOT_REPO}" pull
else:
    !git clone https://github.com/AIpourlapeau/acne04v2 "{ANNOT_REPO}"

import json
annot_path = f'{ANNOT_REPO}/Acne04-v2_annotations.json'
with open(annot_path) as f:
    data = json.load(f)
print(f'✅ Anotasi: {len(data.get("images",[]))} gambar, {len(data.get("annotations",[]))} lesi')

---
## 7. Konfigurasi Path

> **Jalankan cell ini setiap sesi baru Colab** (setelah step 1–6).

In [ ]:
import sys, os, torch
from pathlib import Path

# ── Scripts (dari GitHub repo) ───────────────────────────
SCRIPTS_DIR = Path('/content/acne-seg/scripts')

# ── Dataset ──────────────────────────────────────────────
# Sesuaikan subfolder setelah cek output Step 5
IMAGE_DIR  = Path(KAGGLE_PATH)               # ubah jika subfolder berbeda
ANNOT_JSON = Path('/content/acne04v2/Acne04-v2_annotations.json')

# ── Output → Drive (tidak hilang saat sesi berakhir) ─────
OUTPUT_DIR = Path('/content/drive/MyDrive/acne_project/output')
MASK_DIR   = OUTPUT_DIR / 'refined_masks_output'
TRAIN_DIR  = OUTPUT_DIR / 'training_runs'
INFER_DIR  = OUTPUT_DIR / 'predictions'

for d in [MASK_DIR, TRAIN_DIR, INFER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Python path ──────────────────────────────────────────
for p in [str(SCRIPTS_DIR), str(SCRIPTS_DIR / 'code_dari_claude')]:
    if p not in sys.path:
        sys.path.insert(0, p)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
REFINED_MANIFEST = MASK_DIR / 'refined_manifest.csv'

# ── Validasi ─────────────────────────────────────────────
print('=== PATH CONFIGURATION ===')
print(f'Scripts    : {SCRIPTS_DIR}')
print(f'Images     : {IMAGE_DIR}')
print(f'Annotation : {ANNOT_JSON}')
print(f'Output     : {OUTPUT_DIR}')
print(f'Device     : {DEVICE}')
print()

errors = []
if not SCRIPTS_DIR.exists(): errors.append(f'❌ Scripts: {SCRIPTS_DIR}')
if not IMAGE_DIR.exists():   errors.append(f'❌ Images : {IMAGE_DIR}')
if not ANNOT_JSON.exists():  errors.append(f'❌ Annot  : {ANNOT_JSON}')

if errors:
    for e in errors: print(e)
else:
    n = len(list(IMAGE_DIR.rglob('*.jpg')))
    print(f'✅ Semua path valid | {n} gambar ditemukan')

---
## 8. Generate Refined Mask

**Jalankan sekali saja.** Hasil disimpan ke Drive — tidak perlu diulang di sesi berikutnya.

In [ ]:
import pandas as pd

if REFINED_MANIFEST.exists():
    df = pd.read_csv(REFINED_MANIFEST)
    print(f'ℹ️  Manifest sudah ada ({len(df)} baris) — skip generate.')
    print(df['split'].value_counts().to_string())
else:
    print('Generating refined masks...')
    !python "{SCRIPTS_DIR}/refine_masks_color.py" \
      --image-dir       "{IMAGE_DIR}" \
      --annotation-json "{ANNOT_JSON}" \
      --output-dir      "{MASK_DIR}" \
      --apply-preprocessing \
      --max-images 0

    if REFINED_MANIFEST.exists():
        df = pd.read_csv(REFINED_MANIFEST)
        print(f'\n✅ Manifest dibuat: {len(df)} baris')
        print(df['split'].value_counts().to_string())
    else:
        print('❌ Gagal. Cek error di atas.')

---
## 9. Training

Estimasi: **2–3 jam** di T4 GPU untuk 30+90 epoch.

In [ ]:
OUTPUT_RUN = TRAIN_DIR / 'proposal_hybrid_v1'

!python "{SCRIPTS_DIR}/train_proposal_hybrid_resnet50_malds.py" \
  --manifest        "{REFINED_MANIFEST}" \
  --output-dir      "{OUTPUT_RUN}" \
  --image-size      320 \
  --batch-size      16 \
  --phase1-epochs   30 \
  --phase2-epochs   90 \
  --device          cuda \
  --pretrained \
  --weighted-sampler \
  --num-workers     4 \
  --seed            42

In [ ]:
# ── RESUME jika sesi Colab putus ─────────────────────────
# Uncomment dan jalankan cell ini untuk lanjut training

# !python "{SCRIPTS_DIR}/train_proposal_hybrid_resnet50_malds.py" \
#   --manifest        "{REFINED_MANIFEST}" \
#   --output-dir      "{OUTPUT_RUN}" \
#   --image-size      320 \
#   --batch-size      16 \
#   --phase1-epochs   30 \
#   --phase2-epochs   90 \
#   --device          cuda \
#   --pretrained \
#   --weighted-sampler \
#   --num-workers     4 \
#   --resume

---
## 10. Monitor Progress Training

In [ ]:
import json, matplotlib.pyplot as plt

metrics_path = OUTPUT_RUN / 'metrics.json'
if not metrics_path.exists():
    print('metrics.json belum ada.')
else:
    with open(metrics_path) as f:
        data = json.load(f)
    h = data.get('history', [])
    if not h:
        print('History masih kosong.')
    else:
        ep  = [x['epoch'] for x in h]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        axes[0].plot(ep, [x.get('train_dice',0) for x in h], label='Train')
        axes[0].plot(ep, [x.get('val_dice',0)   for x in h], label='Val')
        axes[0].set_title('Dice Score'); axes[0].legend(); axes[0].grid(alpha=0.3)

        axes[1].plot(ep, [x.get('train_acc',0) for x in h], label='Train')
        axes[1].plot(ep, [x.get('val_acc',0)   for x in h], label='Val')
        axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

        axes[2].plot(ep, [x.get('train_loss',0) for x in h], label='Train')
        axes[2].plot(ep, [x.get('val_loss',0)   for x in h], label='Val')
        axes[2].set_title('Loss'); axes[2].legend(); axes[2].grid(alpha=0.3)

        last = h[-1]
        plt.suptitle(
            f"Epoch {last['epoch']} | Phase: {last.get('phase','?')} | "
            f"Val Dice: {last.get('val_dice',0):.4f} | Val Acc: {last.get('val_acc',0):.4f}"
        )
        plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=100)
        plt.show()

---
## 11. Hasil Test Metrics

In [ ]:
metrics_file = OUTPUT_RUN / 'metrics.json'
if not metrics_file.exists():
    print('Training belum selesai.')
else:
    with open(metrics_file) as f:
        data = json.load(f)
    tm = data.get('test_metrics', {})
    print('=' * 38)
    print('        HASIL TEST SET')
    print('=' * 38)
    print(f"  Accuracy    : {tm.get('acc',   0):.4f}")
    print(f"  Kappa       : {tm.get('kappa', 0):.4f}")
    print(f"  Dice Score  : {tm.get('dice',  0):.4f}")
    print(f"  IoU         : {tm.get('iou',   0):.4f}")
    print('-' * 38)
    print(f"  Seg Loss    : {tm.get('seg_loss', 0):.4f}")
    print(f"  Cls Loss    : {tm.get('cls_loss', 0):.4f}")
    print('=' * 38)
    print(f"  Best Val Loss: {data.get('best_val_loss',0):.4f}")

---
## 12. Inferensi & Visualisasi

In [ ]:
CHECKPOINT = OUTPUT_RUN / 'best_model.pt'
INFER_OUT  = INFER_DIR  / 'proposal_hybrid_v1'

if not CHECKPOINT.exists():
    print('❌ best_model.pt tidak ada. Training harus selesai dulu.')
else:
    !python "{SCRIPTS_DIR}/infer_testing_images.py" \
      --manifest    "{REFINED_MANIFEST}" \
      --checkpoint  "{CHECKPOINT}" \
      --output-dir  "{INFER_OUT}" \
      --split       test \
      --image-size  320 \
      --threshold   0.5 \
      --device      cuda
    print('✅ Inferensi selesai:', INFER_OUT)

In [ ]:
# Tampilkan 8 contoh mask prediksi
import matplotlib.pyplot as plt
from PIL import Image as PILImage

pred_files = sorted(Path(INFER_OUT).glob('*.png'))[:8]
if pred_files:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for ax, pf in zip(axes.flatten(), pred_files):
        ax.imshow(PILImage.open(pf))
        ax.set_title(pf.stem[:20], fontsize=8)
        ax.axis('off')
    plt.suptitle('Predicted Segmentation Masks — Test Set')
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'predicted_masks.png'), dpi=100)
    plt.show()
else:
    print('Belum ada output prediksi.')

---
## Update Script dari GitHub

Kalau kamu sudah push perubahan script ke GitHub, jalankan cell ini untuk pull update ke Colab:

In [ ]:
!git -C /content/acne-seg pull
print('✅ Script berhasil diupdate dari GitHub')